# NonBDNA Finder — Parallel CSV + Comparison + Visualisation Notebook

## Overview
A streamlined **three-phase** workflow:

| Cell | Phase | What it does |
|------|-------|--------------|
| **Cell 1 · Setup** | Configuration | Imports, user config, helper functions, FASTA resolution |
| **Cell 2 · Parallel CSV Generation** | CSV output | Detects Non-B DNA motifs in **all FASTA files in parallel**, writes per-file `motifs.csv`, `class_statistics.csv`, `subclass_statistics.csv` |
| **Cell 3 · Master Comparison** | Aggregation | Reads the per-file CSVs and builds `master_comparison.csv`, `master_class_statistics.csv`, `master_subclass_statistics.csv` |
| **Cell 4 · Simple Visualisations** | Charts | Loads from the saved CSVs only — bar charts and heatmaps |

Run cells **in order**: 1 → 2 → 3 → 4.

---

## Output directory layout
```
<OUTPUT_DIR>/<timestamp>/
├── <file_stem>/
│   ├── motifs.csv                  ← raw motif rows
│   ├── class_statistics.csv        ← per-file class count / density / coverage
│   └── subclass_statistics.csv     ← per-file subclass count / density / coverage
└── _master/
    ├── master_comparison.csv       ← one row per file (summary)
    ├── master_class_statistics.csv ← class stats aggregated across all files
    └── master_subclass_statistics.csv
```


In [ ]:
# =============================================================================
# CELL 1 · SETUP — imports, configuration, helpers
# Edit FASTA_INPUT and OUTPUT_DIR, then run this cell before Cell 2.
# =============================================================================

import sys, os, importlib, glob, gc, time, datetime, re, warnings
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

_REPO_ROOT = os.path.abspath(os.getcwd())
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

# ── Auto-install missing packages ─────────────────────────────────────────────
_REQUIRED = [('psutil','psutil>=5.8'),('pandas','pandas>=1.3'),('numpy','numpy>=1.21'),
             ('matplotlib','matplotlib>=3.5'),('seaborn','seaborn>=0.11'),
             ('tqdm','tqdm>=4.64')]
_miss = [p for m,p in _REQUIRED if importlib.util.find_spec(m) is None]
if _miss:
    import subprocess; subprocess.check_call([sys.executable,'-m','pip','install',*_miss,'-q'])

import pandas as pd
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display, HTML, Image
sns.set_theme(style='whitegrid')
plt.rcParams.update({'xtick.labelsize': 12, 'ytick.labelsize': 12})

# ── Optional fast FASTA parsers ───────────────────────────────────────────────
try:
    import pyfastx as _pyfastx; _HAS_PYFASTX = True
except ImportError:
    _HAS_PYFASTX = False
    try:
        from Bio import SeqIO as _SeqIO; _HAS_SEQIO = True
    except ImportError:
        _HAS_SEQIO = False

# ── USER CONFIGURATION ────────────────────────────────────────────────────────
FASTA_INPUT     = ['*.fna', '*.fasta']   # path, wildcard, or list of patterns
OUTPUT_DIR      = 'notebook_reports'
ENABLED_CLASSES = None                   # None = all; e.g. ['G-Quadruplex','Z-DNA']

# Maximum parallel file workers.  None = number of FASTA files (one thread per file).
MAX_FILE_WORKERS = None

# ── Large-sequence windowed processing ────────────────────────────────────────
LARGE_CHR_THRESHOLD_MB  = 10   # Mb
GENOME_CHUNK_SIZE_MB    = 5    # Mb
GENOME_CHUNK_OVERLAP_KB = 2    # kb

# ── Core imports ─────────────────────────────────────────────────────────────
from Utilities.system_resource_inspector import SystemResourceInspector
from Utilities.adaptive_chunk_planner    import AdaptiveChunkPlanner
from Utilities.nonbscanner               import analyze_sequence as _nbf_analyze

# ── Resolve FASTA files ───────────────────────────────────────────────────────
def _resolve(inp):
    out = []
    for p in ([inp] if isinstance(inp, str) else list(inp)):
        hits = glob.glob(p); out.extend(hits)
        if not hits and os.path.isfile(p): out.append(p)
    return sorted({str(Path(f).resolve()) for f in out})

def _seq_lengths(p):
    L, c = [], 0
    with open(p) as fh:
        for ln in fh:
            s = ln.strip()
            if s.startswith('>'):
                if c: L.append(c); c = 0
            else: c += len(s)
    if c: L.append(c)
    return L

def _stream_fasta(fasta_path):
    """Yield (name, seq_str) tuples, one sequence at a time."""
    if _HAS_PYFASTX:
        for seq in _pyfastx.Fasta(str(fasta_path), build_index=False):
            yield seq.name, seq.seq
    elif _HAS_SEQIO:
        with open(fasta_path) as fh:
            for rec in _SeqIO.parse(fh, 'fasta'):
                yield rec.id, str(rec.seq)
    else:
        name, parts = None, []
        with open(fasta_path) as fh:
            for ln in fh:
                s = ln.rstrip('\n')
                if s.startswith('>'):
                    if name is not None: yield name, ''.join(parts)
                    name = s[1:].split()[0]; parts = []
                else: parts.append(s)
        if name is not None: yield name, ''.join(parts)

FASTA_FILES = _resolve(FASTA_INPUT)
if not FASTA_FILES:
    raise FileNotFoundError(f'No FASTA files found for: {FASTA_INPUT}')

FILE_TYPES = {}
for fp in FASTA_FILES:
    ls = _seq_lengths(fp)
    FILE_TYPES[fp] = ('single' if len(ls)==1 else
                      'multi_equal' if len(set(ls))==1 else 'multi')

print(f'\U0001f4c2  {len(FASTA_FILES)} FASTA file(s) found:')
for fp in FASTA_FILES:
    print(f'   [{FILE_TYPES[fp]:12s}]  {Path(fp).name}')

# ── Adaptive resource planning ────────────────────────────────────────────────
_insp   = SystemResourceInspector()
_budget = _insp.get_memory_budget()
_cpus   = _insp.get_cpu_count()
_total  = max(sum(os.path.getsize(f) for f in FASTA_FILES if os.path.exists(f)), 1_000)
_plan   = AdaptiveChunkPlanner().plan(_total, _budget, _cpus)
CHUNK_SIZE, CHUNK_OVERLAP = _plan['chunk_size'], _plan['overlap']
EXEC_MODE = _plan['mode']
N_FILE_WORKERS = MAX_FILE_WORKERS or len(FASTA_FILES)

_LARGE_CHR_THRESHOLD  = LARGE_CHR_THRESHOLD_MB  * 1_000_000
_GENOME_CHUNK_SIZE    = GENOME_CHUNK_SIZE_MB     * 1_000_000
_GENOME_CHUNK_OVERLAP = GENOME_CHUNK_OVERLAP_KB  * 1_000

_RUN_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
_BASE   = Path(OUTPUT_DIR) / _RUN_TS
_BASE.mkdir(parents=True, exist_ok=True)

# ── Helper functions ─────────────────────────────────────────────────────────
def _scan_sequence(name, seq):
    """Run Non-B DNA detection on a single sequence, handling large sequences."""
    if len(seq) < 10:
        return []
    if len(seq) > _LARGE_CHR_THRESHOLD:
        _step = _GENOME_CHUNK_SIZE - _GENOME_CHUNK_OVERLAP
        _all, _seen = [], {}
        for _ws in range(0, len(seq), _step):
            _we = min(_ws + _GENOME_CHUNK_SIZE, len(seq))
            _wm = _nbf_analyze(sequence=seq[_ws:_we], sequence_name=name,
                               use_chunking=True, chunk_size=CHUNK_SIZE,
                               chunk_overlap=CHUNK_OVERLAP,
                               use_parallel_chunks=(EXEC_MODE=='hybrid'),
                               enabled_classes=ENABLED_CLASSES)
            for _m in _wm:
                _m['Start'] = _m.get('Start', 0) + _ws
                _m['End']   = _m.get('End',   0) + _ws
            _all.extend(_wm)
        for _m in _all:
            _k = (_m.get('Class',''), _m.get('Subclass',''), _m.get('Start',0), _m.get('End',0))
            if _k not in _seen or _m.get('Score',0) > _seen[_k].get('Score',0):
                _seen[_k] = _m
        return sorted(_seen.values(), key=lambda x: x.get('Start', 0))
    return _nbf_analyze(sequence=seq, sequence_name=name, use_chunking=True,
                        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
                        use_parallel_chunks=(EXEC_MODE=='hybrid'),
                        enabled_classes=ENABLED_CLASSES)

def _merge_coverage(intervals, cap=None):
    """Return total merged length of interval list (numpy-based)."""
    if len(intervals) == 0: return 0
    iv = np.array(intervals, dtype=int)
    if cap is not None: iv[:,1] = np.minimum(iv[:,1], cap)
    iv = iv[iv[:,1] > iv[:,0]]
    if len(iv) == 0: return 0
    iv = iv[np.argsort(iv[:,0])]
    s, e = iv[0]; covered = 0
    for cs, ce in iv[1:]:
        if cs <= e: e = max(e, ce)
        else: covered += e - s; s, e = cs, ce
    return covered + e - s

def _gc_and_length(fasta_path):
    """Return (gc_percent, total_bp) for a FASTA file."""
    gc = total = 0
    with open(fasta_path) as fh:
        for ln in fh:
            s = ln.strip()
            if not s or s.startswith('>'): continue
            su = s.upper(); gc += su.count('G') + su.count('C'); total += len(su)
    return (round(gc / total * 100, 2) if total else 0.0), total

def _safe_fname(s): return re.sub(r'[^\w\-]', '_', str(s))

print(f'\u2699\ufe0f  Chunk={CHUNK_SIZE:,}  Overlap={CHUNK_OVERLAP:,}  Mode={EXEC_MODE}  FileWorkers={N_FILE_WORKERS}')
print(f'\U0001f4c2 Run output: {_BASE}')
print('\u2705 Setup complete — run Cell 2 to start parallel CSV generation.')


In [ ]:
# =============================================================================
# CELL 2 · PARALLEL CSV GENERATION
# Processes all FASTA files in parallel (one thread per file).
# Each file produces:
#   • motifs.csv              — raw motif rows
#   • class_statistics.csv    — per-file class stats (count, density, coverage)
#   • subclass_statistics.csv — per-file subclass stats
# =============================================================================

def _process_one_file(fasta_path):
    """
    Detect Non-B DNA motifs in a single FASTA file and write three CSV files.

    Returns a summary dict for Cell 3.
    """
    stem  = Path(fasta_path).stem
    ftype = FILE_TYPES[fasta_path]
    fdir  = _BASE / stem
    fdir.mkdir(parents=True, exist_ok=True)

    _out_csv   = str(fdir / 'motifs.csv')
    _first_row = True
    _csv_cols  = None
    sl_map     = {}          # sequence_name -> length
    total_motifs = 0
    t0 = time.perf_counter()

    for seq_name, seq in _stream_fasta(fasta_path):
        sl_map[seq_name] = len(seq)
        try:
            motifs = _scan_sequence(seq_name, seq)
            if motifs:
                chunk_df = pd.DataFrame(motifs)
                # Normalise dtypes for smaller CSVs
                for _c in ('Start','End','Length'):
                    if _c in chunk_df.columns:
                        chunk_df[_c] = chunk_df[_c].astype('int32')
                if 'Score' in chunk_df.columns:
                    chunk_df['Score'] = chunk_df['Score'].astype('float32')
                if _first_row:
                    _csv_cols = list(chunk_df.columns)
                else:
                    chunk_df = chunk_df.reindex(columns=_csv_cols)
                chunk_df.to_csv(_out_csv, mode='a', index=False,
                                header=_first_row, encoding='utf-8-sig')
                _first_row = False
                total_motifs += len(chunk_df)
                del chunk_df
            del motifs
        except Exception as exc:
            tqdm.write(f'  \u26a0\ufe0f  {stem}/{seq_name[:40]} — skipped ({exc})')
        del seq
        gc.collect()

    elapsed = time.perf_counter() - t0

    # ── Load motifs back (needed to compute coverage) ─────────────────────────
    if total_motifs > 0 and os.path.exists(_out_csv):
        df = pd.read_csv(_out_csv)
        for col, dflt in [('Class','Unknown'),('Subclass','Other'),('Start',0),
                          ('End',0),('Length',0),('Score',0.0),('Strand','+'),
                          ('Sequence_Name','')]:
            if col not in df.columns: df[col] = dflt
        df['Length'] = np.where(df['Length']==0,
                                (df['End']-df['Start']).clip(lower=0), df['Length'])
        df['Source_File'] = Path(fasta_path).name
        df['File_Type']   = ftype
        # Re-save with added columns
        df.to_csv(_out_csv, index=False, encoding='utf-8-sig')
    else:
        df = pd.DataFrame()

    total_bp = max(sum(sl_map.values()), 1)
    gc_pct, _ = _gc_and_length(fasta_path)

    # ── Class statistics CSV ─────────────────────────────────────────────────
    cls_rows = []
    if not df.empty:
        for cls, grp in df.groupby('Class'):
            cov = sum(
                _merge_coverage(sg[['Start','End']].values, cap=sl_map.get(sn, 0))
                for sn, sg in grp.groupby('Sequence_Name')
            )
            cls_rows.append({
                'Class':           cls,
                'Count':           len(grp),
                'Mean_Length':     round(grp['Length'].mean(), 3),
                'Mean_Score':      round(grp['Score'].mean(), 3),
                'Density_per_kb':  round(len(grp) / total_bp * 1000, 4),
                'Coverage_pct':    round(cov / total_bp * 100, 3),
                'Source_File':     Path(fasta_path).name,
            })
    cls_df = pd.DataFrame(cls_rows)
    if not cls_df.empty:
        cls_df = cls_df.sort_values('Count', ascending=False)
    cls_df.to_csv(str(fdir / 'class_statistics.csv'), index=False, encoding='utf-8-sig')

    # ── Subclass statistics CSV ──────────────────────────────────────────────
    sc_rows = []
    if not df.empty:
        for sc, grp in df.groupby('Subclass'):
            cov = sum(
                _merge_coverage(sg[['Start','End']].values, cap=sl_map.get(sn, 0))
                for sn, sg in grp.groupby('Sequence_Name')
            )
            sc_rows.append({
                'Subclass':        sc,
                'Count':           len(grp),
                'Mean_Length':     round(grp['Length'].mean(), 3),
                'Mean_Score':      round(grp['Score'].mean(), 3),
                'Density_per_kb':  round(len(grp) / total_bp * 1000, 4),
                'Coverage_pct':    round(cov / total_bp * 100, 3),
                'Source_File':     Path(fasta_path).name,
            })
    sc_df = pd.DataFrame(sc_rows)
    if not sc_df.empty:
        sc_df = sc_df.sort_values('Count', ascending=False)
    sc_df.to_csv(str(fdir / 'subclass_statistics.csv'), index=False, encoding='utf-8-sig')

    return {
        'stem':           stem,
        'path':           fasta_path,
        'fdir':           str(fdir),
        'file_type':      ftype,
        'seq_lengths':    sl_map,
        'total_motifs':   total_motifs,
        'total_bp':       total_bp,
        'gc_pct':         gc_pct,
        'n_sequences':    len(sl_map),
        'elapsed_s':      round(elapsed, 2),
        'motifs_csv':     str(fdir / 'motifs.csv'),
        'class_csv':      str(fdir / 'class_statistics.csv'),
        'subclass_csv':   str(fdir / 'subclass_statistics.csv'),
    }


# ── Run all files in parallel ─────────────────────────────────────────────────
print(f'\U0001f680 Starting parallel CSV generation for {len(FASTA_FILES)} file(s) '
      f'({N_FILE_WORKERS} worker thread(s))...\n')

_WALL = time.perf_counter()
FILE_RESULTS = {}   # stem -> summary dict

with ThreadPoolExecutor(max_workers=N_FILE_WORKERS) as pool:
    future_map = {pool.submit(_process_one_file, fp): fp for fp in FASTA_FILES}
    for fut in tqdm(as_completed(future_map), total=len(future_map),
                    desc='Files', unit='file'):
        fp = future_map[fut]
        try:
            res = fut.result()
            FILE_RESULTS[res['stem']] = res
            tqdm.write(
                f"  \u2705  {res['stem'][:50]:50s}  "
                f"{res['total_motifs']:>8,} motifs  "
                f"{res['elapsed_s']:.1f}s"
            )
        except Exception as exc:
            tqdm.write(f'  \u26a0\ufe0f  {Path(fp).stem[:50]} — FAILED: {exc}')

_wall_total = time.perf_counter() - _WALL
print(f'\n\u2705 Parallel CSV generation complete in {_wall_total:.1f}s')
print(f'   Files processed: {len(FILE_RESULTS)} / {len(FASTA_FILES)}')
print()
print('Per-file summary:')
for stem, r in FILE_RESULTS.items():
    print(f"  {stem[:45]:45s}  {r['total_motifs']:>8,} motifs  "
          f"{r['total_bp']:>12,} bp  {r['elapsed_s']:.1f}s")
print()
print('\U0001f4be CSV outputs written — run Cell 3 to build the master comparison file.')


In [ ]:
# =============================================================================
# CELL 3 · MASTER COMPARISON FILE
# Reads per-file CSVs written by Cell 2, aggregates them into:
#   • master_comparison.csv        — one summary row per file
#   • master_class_statistics.csv  — class stats across all files
#   • master_subclass_statistics.csv
# =============================================================================

_master_dir = _BASE / '_master'
_master_dir.mkdir(exist_ok=True)

# ── 1. Per-file comparison summary ───────────────────────────────────────────
_cmp_rows = []
for stem, res in FILE_RESULTS.items():
    n = res['total_motifs']
    seq_len = res['total_bp']
    _cmp_rows.append({
        'File':               Path(res['path']).name,
        'Stem':               stem,
        'File_Type':          res['file_type'],
        'Sequences':          res['n_sequences'],
        'Total_bp':           seq_len,
        'GC_Percent':         res['gc_pct'],
        'Total_Motifs':       n,
        'Density_per_kb':     round(n / seq_len * 1000, 4) if seq_len else 0.0,
        'Elapsed_s':          res['elapsed_s'],
    })
    # Enrich with per-file class/subclass CSV columns
    if os.path.exists(res['class_csv']):
        _cls_df = pd.read_csv(res['class_csv'])
        if not _cls_df.empty:
            _cmp_rows[-1]['Classes']     = int(_cls_df['Class'].nunique())
            _cmp_rows[-1]['Coverage_pct'] = round(
                _cls_df['Coverage_pct'].sum() if 'Coverage_pct' in _cls_df.columns else 0.0, 3)
        else:
            _cmp_rows[-1]['Classes'] = 0; _cmp_rows[-1]['Coverage_pct'] = 0.0
    if os.path.exists(res['subclass_csv']):
        _sc_df = pd.read_csv(res['subclass_csv'])
        _cmp_rows[-1]['Subclasses'] = int(_sc_df['Subclass'].nunique()) if not _sc_df.empty else 0
    else:
        _cmp_rows[-1]['Subclasses'] = 0

master_comparison = pd.DataFrame(_cmp_rows)
master_comparison.to_csv(str(_master_dir / 'master_comparison.csv'),
                         index=False, encoding='utf-8-sig')
print('\U0001f4be  master_comparison.csv written')
display(master_comparison)

# ── 2. Master class statistics (concatenate per-file class CSVs) ─────────────
_cls_parts = []
for stem, res in FILE_RESULTS.items():
    if os.path.exists(res['class_csv']):
        _df = pd.read_csv(res['class_csv'])
        if not _df.empty: _cls_parts.append(_df)

if _cls_parts:
    master_class_stats = pd.concat(_cls_parts, ignore_index=True)
    # Add aggregate row for each class across all files
    master_class_agg = (
        master_class_stats.groupby('Class')
        .agg(
            Total_Count   = ('Count',          'sum'),
            Mean_Density  = ('Density_per_kb', 'mean'),
            Mean_Coverage = ('Coverage_pct',   'mean'),
            Mean_Length   = ('Mean_Length',    'mean'),
            Mean_Score    = ('Mean_Score',     'mean'),
            Files_Present = ('Source_File',    'nunique'),
        )
        .round(4)
        .reset_index()
        .sort_values('Total_Count', ascending=False)
    )
    master_class_stats.to_csv(str(_master_dir / 'master_class_statistics.csv'),
                              index=False, encoding='utf-8-sig')
    master_class_agg.to_csv(str(_master_dir / 'master_class_aggregate.csv'),
                            index=False, encoding='utf-8-sig')
    print('\U0001f4be  master_class_statistics.csv + master_class_aggregate.csv written')
    print('\nClass aggregate across all files:')
    display(master_class_agg)
else:
    master_class_stats = pd.DataFrame()
    master_class_agg   = pd.DataFrame()
    print('\u26a0\ufe0f  No class statistics found.')

# ── 3. Master subclass statistics ────────────────────────────────────────────
_sc_parts = []
for stem, res in FILE_RESULTS.items():
    if os.path.exists(res['subclass_csv']):
        _df = pd.read_csv(res['subclass_csv'])
        if not _df.empty: _sc_parts.append(_df)

if _sc_parts:
    master_subclass_stats = pd.concat(_sc_parts, ignore_index=True)
    master_subclass_agg = (
        master_subclass_stats.groupby('Subclass')
        .agg(
            Total_Count   = ('Count',          'sum'),
            Mean_Density  = ('Density_per_kb', 'mean'),
            Mean_Coverage = ('Coverage_pct',   'mean'),
            Mean_Length   = ('Mean_Length',    'mean'),
            Mean_Score    = ('Mean_Score',     'mean'),
            Files_Present = ('Source_File',    'nunique'),
        )
        .round(4)
        .reset_index()
        .sort_values('Total_Count', ascending=False)
    )
    master_subclass_stats.to_csv(str(_master_dir / 'master_subclass_statistics.csv'),
                                 index=False, encoding='utf-8-sig')
    master_subclass_agg.to_csv(str(_master_dir / 'master_subclass_aggregate.csv'),
                               index=False, encoding='utf-8-sig')
    print('\U0001f4be  master_subclass_statistics.csv + master_subclass_aggregate.csv written')
    print('\nSubclass aggregate across all files (top 20):')
    display(master_subclass_agg.head(20))
else:
    master_subclass_stats = pd.DataFrame()
    master_subclass_agg   = pd.DataFrame()
    print('\u26a0\ufe0f  No subclass statistics found.')

print()
print('\u2705 Master comparison files ready — run Cell 4 for visualisations.')
print(f'   Output folder: {_master_dir}')


In [ ]:
# =============================================================================
# CELL 4 · SIMPLE VISUALISATIONS
# Reads the CSV files saved by Cells 2 and 3.  No re-detection needed.
# =============================================================================

def _savefig(fig, path, show=True):
    fig.savefig(str(path), dpi=150, bbox_inches='tight')
    plt.close(fig)
    if show: display(Image(str(path)))


_vis_dir = _master_dir / '_plots'
_vis_dir.mkdir(exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# (A) Per-file class bar charts  (one plot per file)
# ─────────────────────────────────────────────────────────────────────────────
print('=== A. Per-file class bar charts ===')
for stem, res in FILE_RESULTS.items():
    if not os.path.exists(res['class_csv']): continue
    _cls = pd.read_csv(res['class_csv'])
    if _cls.empty: continue
    fig, ax = plt.subplots(figsize=(8, max(3, len(_cls) * 0.45)))
    ax.barh(_cls['Class'][::-1], _cls['Count'][::-1], color='steelblue')
    ax.set_xlabel('Motif Count')
    ax.set_title(f'{stem} — Class Distribution')
    for i, v in enumerate(_cls['Count'][::-1].values):
        ax.text(v + 0.3, i, str(v), va='center', fontsize=8)
    plt.tight_layout()
    _savefig(fig, _vis_dir / f'{_safe_fname(stem)}_class_dist.png', show=False)

print(f'  Saved per-file class bar charts to {_vis_dir}')

# ─────────────────────────────────────────────────────────────────────────────
# (B) Per-file subclass bar charts
# ─────────────────────────────────────────────────────────────────────────────
print('=== B. Per-file subclass bar charts ===')
for stem, res in FILE_RESULTS.items():
    if not os.path.exists(res['subclass_csv']): continue
    _sc = pd.read_csv(res['subclass_csv'])
    if _sc.empty: continue
    fig, ax = plt.subplots(figsize=(8, max(4, len(_sc) * 0.4)))
    ax.barh(_sc['Subclass'][::-1], _sc['Count'][::-1], color='darkorange')
    ax.set_xlabel('Motif Count')
    ax.set_title(f'{stem} — Subclass Distribution')
    plt.tight_layout()
    _savefig(fig, _vis_dir / f'{_safe_fname(stem)}_subclass_dist.png', show=False)

print(f'  Saved per-file subclass bar charts to {_vis_dir}')

# ─────────────────────────────────────────────────────────────────────────────
# (C) Global class distribution (all files combined)
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== C. Global class distribution ===')
if not master_class_agg.empty:
    fig, ax = plt.subplots(figsize=(8, max(3, len(master_class_agg) * 0.45)))
    ax.barh(master_class_agg['Class'][::-1], master_class_agg['Total_Count'][::-1],
            color='steelblue')
    ax.set_xlabel('Total Motif Count (all files)')
    ax.set_title('Global Class Distribution')
    for i, v in enumerate(master_class_agg['Total_Count'][::-1].values):
        ax.text(v + 0.3, i, str(v), va='center', fontsize=8)
    plt.tight_layout()
    _savefig(fig, _vis_dir / 'global_class_distribution.png')

# ─────────────────────────────────────────────────────────────────────────────
# (D) Global subclass distribution (all files combined)
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== D. Global subclass distribution ===')
if not master_subclass_agg.empty:
    _top_sc = master_subclass_agg.head(25)
    fig, ax = plt.subplots(figsize=(8, max(4, len(_top_sc) * 0.4)))
    ax.barh(_top_sc['Subclass'][::-1], _top_sc['Total_Count'][::-1], color='darkorange')
    ax.set_xlabel('Total Motif Count (all files)')
    ax.set_title('Global Subclass Distribution (top 25)')
    plt.tight_layout()
    _savefig(fig, _vis_dir / 'global_subclass_distribution.png')

# ─────────────────────────────────────────────────────────────────────────────
# (E) Density comparison across files
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== E. Density comparison across files ===')
if not master_comparison.empty:
    _mc = master_comparison.sort_values('Density_per_kb', ascending=False)
    fig, ax = plt.subplots(figsize=(max(6, len(_mc) * 1.4), 4))
    bars = ax.bar(_mc['Stem'].apply(lambda x: x[:25]), _mc['Density_per_kb'], color='teal')
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
    ax.set_ylabel('Motifs per kb')
    ax.set_title('Motif Density Comparison Across Files')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    _savefig(fig, _vis_dir / 'density_comparison.png')

# ─────────────────────────────────────────────────────────────────────────────
# (F) Coverage comparison across files
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== F. Coverage comparison across files ===')
if not master_comparison.empty and 'Coverage_pct' in master_comparison.columns:
    _mc2 = master_comparison.sort_values('Coverage_pct', ascending=False)
    fig, ax = plt.subplots(figsize=(max(6, len(_mc2) * 1.4), 4))
    bars = ax.bar(_mc2['Stem'].apply(lambda x: x[:25]), _mc2['Coverage_pct'],
                  color='mediumseagreen')
    ax.bar_label(bars, fmt='%.2f%%', padding=2, fontsize=8)
    ax.set_ylabel('Coverage (%)')
    ax.set_title('Non-B DNA Coverage Comparison Across Files')
    ax.set_ylim(0, min(100, _mc2['Coverage_pct'].max() * 1.15 + 0.5))
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    _savefig(fig, _vis_dir / 'coverage_comparison.png')

# ─────────────────────────────────────────────────────────────────────────────
# (G) Class density heatmap (files × classes)
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== G. Class density heatmap (files × classes) ===')
if not master_class_stats.empty and len(FILE_RESULTS) >= 1:
    _piv = (master_class_stats
            .pivot_table(index='Source_File', columns='Class',
                         values='Density_per_kb', fill_value=0))
    if not _piv.empty:
        fig, ax = plt.subplots(figsize=(max(10, len(_piv.columns) * 1.2),
                                        max(4,  len(_piv) * 0.7)))
        sns.heatmap(_piv, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
                    linewidths=0.4, cbar_kws={'label': 'Motifs per kb'})
        ax.set_title('Class Density Heatmap (motifs/kb) — Files × Classes')
        ax.set_xlabel('Non-B Class'); ax.set_ylabel('File')
        plt.tight_layout()
        _savefig(fig, _vis_dir / 'class_density_heatmap.png')

# ─────────────────────────────────────────────────────────────────────────────
# (H) Class coverage heatmap (files × classes)
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== H. Class coverage heatmap (files × classes) ===')
if not master_class_stats.empty:
    _piv2 = (master_class_stats
             .pivot_table(index='Source_File', columns='Class',
                          values='Coverage_pct', fill_value=0))
    if not _piv2.empty:
        fig, ax = plt.subplots(figsize=(max(10, len(_piv2.columns) * 1.2),
                                        max(4,  len(_piv2) * 0.7)))
        sns.heatmap(_piv2, annot=True, fmt='.3f', cmap='Blues', ax=ax,
                    linewidths=0.4, cbar_kws={'label': 'Coverage %'})
        ax.set_title('Class Coverage Heatmap (%) — Files × Classes')
        ax.set_xlabel('Non-B Class'); ax.set_ylabel('File')
        plt.tight_layout()
        _savefig(fig, _vis_dir / 'class_coverage_heatmap.png')

# ─────────────────────────────────────────────────────────────────────────────
# (I) Mean density by class (bar — global aggregate)
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== I. Mean class density (global aggregate) ===')
if not master_class_agg.empty:
    _cd = master_class_agg.sort_values('Mean_Density', ascending=False)
    fig, ax = plt.subplots(figsize=(8, max(3, len(_cd) * 0.45)))
    ax.barh(_cd['Class'][::-1], _cd['Mean_Density'][::-1], color='teal')
    ax.set_xlabel('Mean Motifs per kb (across files)')
    ax.set_title('Mean Class Density — All Files')
    for i, v in enumerate(_cd['Mean_Density'][::-1].values):
        ax.text(v, i, f'{v:.4f}', va='center', fontsize=8)
    plt.tight_layout()
    _savefig(fig, _vis_dir / 'global_class_density.png')

# ─────────────────────────────────────────────────────────────────────────────
# (J) Mean coverage by class (bar — global aggregate)
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== J. Mean class coverage (global aggregate) ===')
if not master_class_agg.empty:
    _cc = master_class_agg.sort_values('Mean_Coverage', ascending=False)
    fig, ax = plt.subplots(figsize=(8, max(3, len(_cc) * 0.45)))
    ax.barh(_cc['Class'][::-1], _cc['Mean_Coverage'][::-1], color='mediumseagreen')
    ax.set_xlabel('Mean Coverage % (across files)')
    ax.set_title('Mean Class Coverage (%) — All Files')
    for i, v in enumerate(_cc['Mean_Coverage'][::-1].values):
        ax.text(v, i, f'{v:.3f}%', va='center', fontsize=8)
    plt.tight_layout()
    _savefig(fig, _vis_dir / 'global_class_coverage.png')

print()
print(f'\u2705 All visualisations saved to {_vis_dir}')
print(f'   Master CSV files in {_master_dir}')
